In [0]:
# =============================================================================
# SILVER LAYER - DATA CLEANING & VALIDATION
# =============================================================================
# Purpose: Clean, validate, and deduplicate data from Bronze layer
# Input: cryptocatalog.crypto_raw.bronze_ohlcv_1min
# Output: cryptocatalog.crypto_refined.silver_ohlcv_1min
# =============================================================================

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

# Configuration
catalog_name = "cryptocatalog"
bronze_table = f"{catalog_name}.crypto_raw.bronze_ohlcv_1min"
silver_table = f"{catalog_name}.crypto_refined.silver_ohlcv_1min"

print("🥈 SILVER LAYER CONFIGURATION")
print("=" * 80)
print(f"Input Table:  {bronze_table}")
print(f"Output Table: {silver_table}")
print("=" * 80)

# Create Unity Catalog schemas if not exists
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.crypto_refined")
    spark.sql(f"ALTER SCHEMA {catalog_name}.crypto_refined SET DBPROPERTIES ('layer' = 'silver', 'description' = 'Cleaned and validated crypto data')")
    print(f"\n✅ Schema ready: {catalog_name}.crypto_refined")
except Exception as e:
    print(f"⚠️ Schema setup: {e}")

In [0]:
def transform_to_silver():
    """
    Silver Layer: Clean, validate, and deduplicate data
    - Type casting and column renaming
    - Data quality filters (price > 0, valid ranges)
    - Deduplication (keep latest record per timestamp)
    - Add calculated fields (price_change, price_range, etc.)
    """
    print("=" * 80)
    print("🥈 SILVER LAYER - DATA CLEANING & VALIDATION")
    print("=" * 80)
    
    # Read from bronze
    df_bronze = spark.table(bronze_table)
    
    input_count = df_bronze.count()
    print(f"📥 Input records from Bronze: {input_count:,}")
    
    # Data transformations - Type casting and renaming
    df_silver = df_bronze.select(
        # Time columns
        F.col("time").cast("bigint").alias("timestamp_unix"),
        F.from_unixtime(F.col("time")).cast("timestamp").alias("timestamp"),
        F.to_date(F.from_unixtime(F.col("time"))).alias("date"),
        
        # OHLCV columns with proper types
        F.col("open").cast("double").alias("open_price"),
        F.col("high").cast("double").alias("high_price"),
        F.col("low").cast("double").alias("low_price"),
        F.col("close").cast("double").alias("close_price"),
        F.col("volumefrom").cast("double").alias("volume_btc"),
        F.col("volumeto").cast("double").alias("volume_usd"),
        
        # Metadata
        F.col("source_system"),
        F.col("ingestion_timestamp"),
        
        # Add symbol and currency
        F.lit("BTC").alias("symbol"),
        F.lit("USD").alias("quote_currency")
    )
    
    # Data quality checks
    df_silver = df_silver \
        .filter(F.col("close_price") > 0) \
        .filter(F.col("volume_usd") >= 0) \
        .filter(F.col("high_price") >= F.col("low_price")) \
        .filter(F.col("close_price").between(F.col("low_price"), F.col("high_price")))
    
    after_quality = df_silver.count()
    print(f"✅ After quality checks: {after_quality:,}")
    
    # Deduplicate (keep latest ingestion per timestamp)
    window_spec = Window.partitionBy("timestamp_unix").orderBy(F.desc("ingestion_timestamp"))
    df_silver = df_silver \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num")
    
    after_dedup = df_silver.count()
    print(f"✅ After deduplication: {after_dedup:,}")
    
    # Add calculated fields
    df_silver = df_silver.select(
        "*",
        (F.col("close_price") - F.col("open_price")).alias("price_change"),
        ((F.col("close_price") - F.col("open_price")) / F.col("open_price") * 100).alias("price_change_pct"),
        (F.col("high_price") - F.col("low_price")).alias("price_range"),
        F.when(F.col("close_price") >= F.col("open_price"), True).otherwise(False).alias("is_green_candle"),
        F.current_timestamp().alias("processed_timestamp")
    )
    
    # Write to silver table
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(silver_table)
    
    output_count = df_silver.count()
    print(f"\n✅ Processed {output_count:,} clean records to {silver_table}")
    
    # Data quality summary
    quality_stats = df_silver.agg(
        F.min("close_price").alias("min_price"),
        F.max("close_price").alias("max_price"),
        F.avg("close_price").alias("avg_price"),
        F.sum("volume_usd").alias("total_volume_usd"),
        F.count("*").alias("record_count")
    ).collect()[0]
    
    print("\n📊 Data Quality Summary:")
    print(f"  • Price Range: ${quality_stats['min_price']:,.2f} - ${quality_stats['max_price']:,.2f}")
    print(f"  • Average Price: ${quality_stats['avg_price']:,.2f}")
    print(f"  • Total Volume: ${quality_stats['total_volume_usd']:,.0f}")
    print(f"  • Records: {quality_stats['record_count']:,}")
    
    return df_silver

# Run silver transformation
df_silver = transform_to_silver()

if df_silver:
    print("\n" + "="*80)
    print("📋 Sample Silver Data:")
    print("="*80)
    df_silver.orderBy(F.desc("timestamp")).limit(10).show(truncate=False)

In [0]:
# Verify silver table has no duplicates
df_check = spark.table(silver_table)

total_count = df_check.count()
distinct_timestamp = df_check.select("timestamp_unix").distinct().count()
duplicates = total_count - distinct_timestamp

print("=" * 80)
print("📊 SILVER TABLE STATISTICS")
print("=" * 80)
print(f"Total Rows:             {total_count:,}")
print(f"Distinct Timestamps:    {distinct_timestamp:,}")
print(f"Duplicates:             {duplicates:,}")

if duplicates == 0:
    print(f"\n✅ PERFECT! Silver has NO duplicates (as expected)")
else:
    print(f"\n⚠️ WARNING: Silver should have NO duplicates!")
    
print("=" * 80)